In [1]:
import sys
sys.path.append('../../../../../')
%matplotlib inline

In [2]:
import numpy as np
from CADETProcess.processModel import (
    ComponentSystem, LumpedRateModelWithPores, StericMassAction,
)
from CADETProcess.instruments import LCFlowSheet, PulseInjection, LWE
from CADETProcess.comparison import Comparator
from CADETProcess.reference import ReferenceIO
from CADETProcess.characterization import (
    setup_comparators,
    CharacterizeBed,
    CharacterizeParticles,
    CharacterizeAdsorptionParameters,
)

[INFO 08-12 16:17:57] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.52


In [3]:
cs = ComponentSystem(["Salt"])
Q = 8.3e-9  # m³/s

fs = LCFlowSheet(
    cs,
    sample_loop_volume=50e-9,
    ColumnModel=LumpedRateModelWithPores,
)
process = PulseInjection(
    "bed_characterization",
    fs,
    c_buffer_a=[100.0],
    c_sample=[0.0],
    cycle_time=600.0,
    flow_rate=Q,
)

In [4]:
# Synthetic "experimental" data for illustration
time = np.linspace(0, 600, 601)
solution = np.exp(-0.5 * ((time - 300) / 30) ** 2).reshape(-1, 1)
reference = ReferenceIO("uv_280", time, solution, flow_rate=Q)
reference.component_system = cs

In [5]:
comparator = Comparator("bed_characterization")
comparator.add_reference(reference)
comparator.add_difference_metric(
    "Shape",
    reference,
    solution_path="column.outlet.outlet[0]",
)

C:\Users\ezeiry\AppData\Local\Temp\ipykernel_20172\2324938077.py:2: DeprecationWarning: add_reference() is deprecated and will be removed in v1.0. Pass a pre-constructed metric instance to add_difference_metric() instead: metric = SSE(reference); comparator.add_difference_metric(metric, solution_path)
  comparator.add_reference(reference)
C:\Users\ezeiry\AppData\Local\Temp\ipykernel_20172\2324938077.py:3: DeprecationWarning: Passing a metric class name as a string to add_difference_metric() is deprecated and will be removed in v1.0. Construct the metric directly and pass the instance instead: metric = SSE(reference); comparator.add_difference_metric(metric, solution_path)
  comparator.add_difference_metric(


In [ ]:
from CADETProcess.simulator import Cadet
from CADETProcess.optimization import U_NSGA3

simulator = Cadet()

char = CharacterizeBed("bed", process, comparator, simulator)

optimizer = U_NSGA3()
optimizer.optimize(char)

In [ ]:
from CADETProcess.characterization import CharacterizeTubing

char = CharacterizeTubing(
    "tubing", process, "tubing_post_column", comparator, simulator
)
print(char.variable_names)
# ['tubing_post_column_length', 'tubing_post_column_axial_dispersion']

In [ ]:
from CADETProcess.characterization import CharacterizePreInjection

char = CharacterizePreInjection("pre_inj", process, comparator, simulator)
print(char.variable_names)
# ['tubing_pre_injection_length', 'mixer_volume']

In [ ]:
char = CharacterizeBed("bed", process, comparator, simulator)
print(char.variable_names)
# ['bed_porosity', 'axial_dispersion']

In [ ]:
from CADETProcess.characterization import CharacterizeParticles

char = CharacterizeParticles(
    "particles", process, comparator, simulator,
    include_particle_porosity=True,
    include_film_diffusion=True,
    component_index=0,
)
print(char.variable_names)
# ['particle_porosity', 'film_diffusion']

In [ ]:
from CADETProcess.characterization import CharacterizeCapacity

char = CharacterizeCapacity("capacity", process, comparator, simulator)
print(char.variable_names)
# ['capacity']

In [ ]:
from CADETProcess.processModel import StericMassAction
from CADETProcess.characterization import CharacterizeAdsorptionParameters

cs2 = ComponentSystem(["Salt", "Protein"])
fs2 = LCFlowSheet(
    cs2,
    sample_loop_volume=50e-9,
    ColumnModel=LumpedRateModelWithPores,
    BindingModel=StericMassAction,
)
lwe = LWE(
    "lwe", fs2,
    c_buffer_a=[20.0, 0.0], c_buffer_b=[1000.0, 0.0], c_sample=[20.0, 0.5],
    delta_t_wash=120.0, delta_t_elute=600.0, delta_t_final_wash=120.0,
    flow_rate_wash=Q,
)

char = CharacterizeAdsorptionParameters(
    "sma", lwe, comparator, simulator,
    is_kinetic=False,
    component_index=1,
)
print(char.variable_names)
# ['characteristic_charge', 'adsorption_rate']

In [ ]:
processes = [setup_lwe(cv) for cv in [4, 8, 12, 16]]  # one LWE per gradient length
references = [load_reference(cv) for cv in [4, 8, 12, 16]]
start_times = [peak_start[cv] for cv in [4, 8, 12, 16]]
end_times   = [peak_end[cv]   for cv in [4, 8, 12, 16]]

comparators = setup_comparators(
    processes, references,
    solution_path="tubing_post_column.outlet",
    metrics=["Shape"],
    components=["Protein"],
    start=start_times,
    end=end_times,
)

char = CharacterizeAdsorptionParameters(
    "sma_multi", processes, comparators, simulator,
    component_index=1,
)